To demonstrate the ``ShapModel().add_feat_impact()`` method, we obtain the DOM_GSEC example dataset and its respective feature set (see [Breimann25]_):

In [1]:
import aaanalysis as aa
aa.options["verbose"] = False # Disable verbosity

df_seq = aa.load_dataset(name="DOM_GSEC", n=3)
labels = df_seq["label"].to_list()
df_feat = aa.load_features(name="DOM_GSEC").head(5)

# Create feature matrix
sf = aa.SequenceFeature()
df_parts = sf.get_df_parts(df_seq=df_seq)
X = sf.feature_matrix(features=df_feat["feature"], df_parts=df_parts)

aa.display_df(df_seq)

/Users/stephanbreimann/Programming/1Packages/wt-410-dataset-metadata/aaanalysis/feature_engineering/_backend/cpp_run.py:164: UserWarning: CPP is using the Python kernel fallback — the compiled Cython extension is not available in this install. Output is bit-exact with the Cython path but ~2x slower. Reinstall via `pip install --force-reinstall aaanalysis` to fetch a prebuilt wheel.
  warnings.warn(


,entry,gene,sequence,label,tmd_start,tmd_stop,jmd_n,tmd,jmd_c
1,Q14802,FXYD3,MQKVTLGLLVFLAGF...PGETPPLITPGSAQS,0,37,59,NSPFYYDWHS,LQVGGLICAGVLCAMGIIIVMSA,KCKCKFGQKS
2,Q86UE4,MTDH,MAARSWQDELAQQAE...SPKQIKKKKKARRET,0,50,72,LGLEPKRYPG,WVILVGTGALGLLLLFLLGYGWA,AACAGARKKR
3,Q969W9,PMEPA1,MHRLMGVNSTAAAAA...AIWSKEKDKQKGHPL,0,41,63,FQSMEITELE,FVQIIIIVVVMMVMVVVITCLLS,HYKLSARSFI
4,P05067,APP,MLPGLALLLLAAWTA...GYENPTYKFFEQMQN,1,701,723,FAEDVGSNKG,AIIGLMVGGVVIATVIVITLVML,KKKQYTSIHH
5,P14925,Pam,MAGRARSGLLLLLLG...EEEYSAPLPKPAPSS,1,868,890,KLSTEPGSGV,SVVLITTLLVIPVLVLLAIVMFI,RWKKSRAFGD
6,P70180,Npr3,MRSLLLFTFSACVLL...RELREDSIRSHFSVA,1,477,499,PCKSSGGLEE,SAVTGIVVGALLGAGLLMAFYFF,RKKYRITIER


We can now create a ``ShapModel`` object and fit it to create the ``shap_values``, which are saved internally:

In [2]:
sm = aa.ShapModel()
sm.fit(X, labels=labels)

shap_values = sm.shap_values

# Print SHAP values and expected value
print("SHAP values explain the feature impact for 3 negative and 3 positive samples")
print(shap_values.round(2))

SHAP values explain the feature impact for 3 negative and 3 positive samples
[[-0.1  -0.11 -0.08 -0.1  -0.08]
 [-0.12 -0.12 -0.08 -0.1  -0.08]
 [-0.14 -0.14 -0.04 -0.09 -0.02]
 [ 0.13  0.13  0.06  0.09  0.05]
 [ 0.12  0.12  0.08  0.1   0.08]
 [ 0.12  0.13  0.08  0.1   0.07]]


We can now include the feature impact (i.e., SHAP values normalized such that their absolute values sum up to 100%) by providing ``df_feat`` to the ``ShapModel().add_feat_impact()`` method:

In [3]:
# Add feature impact of each sample (Protein0 to Protein5)
df_feat = sm.add_feat_impact(df_feat=df_feat)
aa.display_df(df_feat)

,feature,category,subcategory,scale_name,scale_description,abs_auc,abs_mean_dif,mean_dif,std_test,std_ref,p_val_mann_whitney,p_val_fdr_bh,positions,feat_importance,feat_importance_std,feat_impact_Protein0,feat_impact_Protein1,feat_impact_Protein2,feat_impact_Protein3,feat_impact_Protein4,feat_impact_Protein5
1,"TMD_C_JMD_C-Seg...3,4)-KLEP840101",Energy,Charge,Charge,"Net charge (Kle...n et al., 1984)",0.244000,0.103666,0.103666,0.106692,0.110506,0.000000,0.000000,"31,32,33,34,35",0.970400,1.438918,-22.140000,-23.610000,-31.390000,28.180000,24.120000,24.730000
2,"TMD_C_JMD_C-Seg...3,4)-FINA910104",Conformation,α-helix (C-cap),α-helix termination,"Helix terminati...n et al., 1991)",0.243000,0.085064,0.085064,0.098774,0.096946,0.000000,0.000000,"31,32,33,34,35",0.000000,0.000000,-23.270000,-24.430000,-32.920000,28.700000,24.940000,25.570000
3,"TMD_C_JMD_C-Seg...6,9)-LEVM760105",Shape,Side chain length,Side chain length,"Radius of gyrat... (Levitt, 1976)",0.233000,0.137044,0.137044,0.161683,0.176964,0.000000,0.000001,"32,33",1.554800,2.109848,-16.590000,-16.130000,-9.030000,12.220000,15.370000,15.390000
4,"TMD_C_JMD_C-Seg...3,4)-HUTJ700102",Energy,Entropy,Entropy,"Absolute entrop...Hutchens, 1970)",0.229000,0.098224,0.098224,0.106865,0.124608,0.000000,0.000001,"31,32,33,34,35",3.111200,3.109955,-21.090000,-20.220000,-21.240000,20.330000,19.220000,19.520000
5,"TMD_C_JMD_C-Seg...6,9)-RADA880106",ASA/Volume,Volume,Accessible surface area (ASA),"Accessible surf...olfenden, 1988)",0.223000,0.095071,0.095071,0.114758,0.132829,0.000000,0.000002,"32,33",0.000000,0.000000,-16.910000,-15.600000,-5.420000,10.560000,16.350000,14.790000


To include the impact of a specific sample, use the ``samples`` parameter indicating the position index of the sample within the ``shap_values`` attribute (the same as in the ``labels`` provided to the ``ShapModel().fit()`` method). You need to set ``drop=True`` to override the feature impact columns:

In [4]:
# First protein
df_feat = sm.add_feat_impact(df_feat=df_feat, drop=True, samples=0)
aa.display_df(df_feat, n_cols=-1)

,feat_impact_Protein0
1,-22.140000
2,-23.270000
3,-16.590000
4,-21.090000
5,-16.910000


You can provide a specific ``names`` for the corresponding sample:

In [5]:
# Single sample
df_feat = sm.add_feat_impact(df_feat=df_feat, drop=True, samples=0, names="Selected_sample")
aa.display_df(df_feat, n_cols=-1)

,feat_impact_Selected_sample
1,-22.140000
2,-23.270000
3,-16.590000
4,-21.090000
5,-16.910000


Samples can also be selected by accession instead of position. Provide ``df_seq`` and pass entry name(s) to ``samples``; they are resolved to the matching row(s), and the impact column is named after the accession unless ``names`` is given:

In [6]:
# Select a sample by its entry/accession via df_seq, labelling the impact column by the
# readable gene name from load_dataset's bundled 'gene' column.
row = df_seq.iloc[0]
df_feat = sm.add_feat_impact(df_feat=df_feat, drop=True, df_seq=df_seq,
                             samples=row["entry"], names=row["gene"])
aa.display_df(df_feat, n_cols=-1)

,feat_impact_FXYD3
1,-22.140000
2,-23.270000
3,-16.590000
4,-21.090000
5,-16.910000


**Computing feature impact**

Three different scenarios are possible:

a) **Single sample**: Compute the feature impact for a single sample (above).
b) **Multiple samples**: Compute the feature impact for multiple samples (all by default).
c) **Group of samples**: Compute the average feature impact and standard deviation for a group.

To focus on specific samples, specify their indices in ``samples``. If ``names`` is provided, its length should match ``samples``.

In [7]:
# Multiple samples
df_feat = sm.add_feat_impact(df_feat=df_feat, drop=True, samples=[0, 1], names=["Sample 1", "Sample 2"])
aa.display_df(df_feat, n_cols=-2)

,feat_impact_Sample 1,feat_impact_Sample 2
1,-22.140000,-23.610000
2,-23.270000,-24.430000
3,-16.590000,-16.130000
4,-21.090000,-20.220000
5,-16.910000,-15.600000


To calculate the group average, set ``group_average=True`` and specify the sample indices in `sample_positions`. Provide a ``names`` for the group, or 'Group' will be used by default:

In [8]:
# Group of samples
df_feat = sm.add_feat_impact(df_feat=df_feat, drop=True, samples=[0, 1], group_average=True)
aa.display_df(df_feat, n_cols=-2)

,feat_impact_Group,feat_impact_std_Group
1,-22.900000,1.574111
2,-23.870000,1.456949
3,-16.350000,0.372252
4,-20.640000,0.326228
5,-16.230000,0.057294


Setting ``shap_feat_importance=True``, will compute the SHAP value-based feature importance:

In [9]:
# SHAP value-based feature importance
df_feat = sm.add_feat_impact(df_feat=df_feat, drop=True, shap_feat_importance=True)
aa.display_df(df_feat, n_cols=-1)

,feat_importance
1,25.570000
2,26.510000
3,14.230000
4,20.240000
5,13.440000


**Further parameters.** ``add_feat_impact`` also accepts ``normalize`` (default ``True``): when ``True`` the impact is scaled so its absolute values sum to 100%; set ``normalize=False`` to keep the raw SHAP values as the impact.

In [10]:
# Further parameters: normalize=False keeps the raw SHAP values instead of scaling the
# absolute feature impact to sum to 100%.
df_feat = sm.add_feat_impact(df_feat=df_feat, drop=True, samples=0, normalize=False)
aa.display_df(df_feat, n_cols=-1)

,feat_impact_Protein0
1,-22.143347
2,-23.269386
3,-16.590666
4,-21.085189
5,-16.911412
